# B08 — ProofWriter eval (`/predict`)

Evaluates our Type 1 pipeline on **ProofWriter** ([`D3xter1922/proofwriter-dataset`](https://huggingface.co/datasets/D3xter1922/proofwriter-dataset)).

Each row is seq2seq:
- `translation.en` = `"$answer$ ; $proof$ ; $question$ = <stmt> ; $context$ = sent1: ... sentN: ..."`
- `translation.ro` = `"$answer$ = True ; $proof$ = sent4"`

**Mapping to our schema:** context sentences → `premises`; the `$question$` statement → a polar YNU query; gold `$answer$` `True/False/Unknown` → `Yes/No/Uncertain`; gold `$proof$` `sentK` → 0-based `premises_used` (`K-1`).

**Scoring:** `sample_score = 0.5·P1 + 0.5·P2`. P1 = answer correct. P2 = F1 over premise-index sets (proof granularity differs from our Z3 core, so P2 is approximate).

In [12]:
import json, re, time, asyncio, statistics, random, urllib.request, urllib.parse, urllib.error
from pathlib import Path
from collections import Counter

import httpx
import pandas as pd

API_BASE    = "https://api.iamphuckhang.dev"
PREDICT_URL = f"{API_BASE}/predict"

HF_DATASET = "D3xter1922/proofwriter-dataset"
HF_CONFIG  = "default"
HF_SPLIT   = "test"
N_SAMPLES  = 100          # rows to evaluate (set None / >= split size for full)
SEED       = 2026           # reproducible random sample across the split
CONCURRENCY = 6
TIMEOUT     = 60.0

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
OUTPUT_JSON = ROOT / f"outputs/B08_proofwriter_eval_{time.strftime('%Y%m%d_%H%M%S')}.json"
print("dataset :", HF_DATASET, HF_SPLIT, "| N =", N_SAMPLES, "| seed =", SEED)
print("output  :", OUTPUT_JSON)
print("endpoint:", PREDICT_URL)

dataset : D3xter1922/proofwriter-dataset test | N = 100 | seed = 2026
output  : /home/phuckhang/MyWorkspace/Exact2026/outputs/B08_proofwriter_eval_20260621_070848.json
endpoint: https://api.iamphuckhang.dev/predict


## Load + parse ProofWriter rows (via HF datasets-server)

In [13]:
ANSWER_MAP = {"true": "Yes", "false": "No", "unknown": "Uncertain"}
_DS = "https://datasets-server.huggingface.co"
_UA = {"User-Agent": "Mozilla/5.0"}
DATA_DIR = ROOT / "src/exact/datasets/proofwriter"

def parquet_urls() -> list[str]:
    """Resolve the parquet shard URLs for this split (one cheap API call)."""
    d = json.load(urllib.request.urlopen(
        urllib.request.Request(f"{_DS}/parquet?dataset={urllib.parse.quote(HF_DATASET)}",
                               headers=_UA), timeout=60))
    return [f["url"] for f in d["parquet_files"]
            if f["split"] == HF_SPLIT and f["config"] == HF_CONFIG]

def load_split_df() -> pd.DataFrame:
    """Download the split's parquet shard(s) once (HF CDN — no rate limit) and
    read them locally; later runs reuse the cached files."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    frames = []
    for i, url in enumerate(parquet_urls()):
        local = DATA_DIR / f"{HF_SPLIT}_{i:04d}.parquet"
        if not local.exists():
            req = urllib.request.Request(url, headers=_UA)
            with urllib.request.urlopen(req, timeout=120) as resp:
                local.write_bytes(resp.read())
            print(f"  downloaded {local.name} ({local.stat().st_size // 1024} KB)")
        frames.append(pd.read_parquet(local))
    return pd.concat(frames, ignore_index=True)

def parse_en(en: str) -> tuple[str, list[str]]:
    q = re.search(r"\$question\$ = (.*?) ; \$context\$ =", en, re.S)
    c = re.search(r"\$context\$ = (.*)$", en, re.S)
    question = q.group(1).strip() if q else ""
    sents = []
    if c:
        for m in re.finditer(r"sent\d+:\s*(.*?)(?=\s*sent\d+:|$)", c.group(1).strip(), re.S):
            sents.append(m.group(1).strip())
    return question, sents

def parse_ro(ro: str) -> tuple[str | None, list[int]]:
    a = re.search(r"\$answer\$ = (\w+)", ro)
    proof = sorted({int(x) - 1 for x in re.findall(r"sent(\d+)", ro)})
    return (a.group(1).lower() if a else None), proof

df = load_split_df()
data = [{"en": t["en"], "ro": t["ro"]} for t in df["translation"]]
print(f"split loaded: {len(data)} rows from {DATA_DIR}")

if N_SAMPLES is None or N_SAMPLES >= len(data):
    rows = data
else:
    random.seed(SEED)
    rows = random.sample(data, N_SAMPLES)

samples = []
for i, t in enumerate(rows):
    question, sents = parse_en(t["en"])
    gold_raw, proof = parse_ro(t["ro"])
    if not question or not sents or gold_raw not in ANSWER_MAP:
        continue
    samples.append({
        "query_id": f"PW_{i:04d}",
        "type": "type1",
        "query": f"According to the premises, is the following statement true? {question}",
        "premises": sents,
        "options": None,
        "_gold": ANSWER_MAP[gold_raw],
        "_gold_raw": gold_raw,
        "_gold_premises": proof,
        "_statement": question,
    })

all_samples = samples
print(f"parsed {len(samples)}/{len(rows)} rows | gold dist: {dict(Counter(s['_gold'] for s in samples))}")
print("\nexample:")
ex = samples[0]
print(json.dumps({k: ex[k] for k in ('query', 'premises', '_gold', '_gold_premises')},
                 indent=2)[:700])

split loaded: 11820 rows from /home/phuckhang/MyWorkspace/Exact2026/src/exact/datasets/proofwriter
parsed 100/100 rows | gold dist: {'No': 47, 'Yes': 53}

example:
{
  "query": "According to the premises, is the following statement true? Gary is not white.",
  "premises": [
    "Anne is round.",
    "Bob is big.",
    "Bob is round.",
    "Bob is white.",
    "Fiona is big.",
    "Fiona is green.",
    "Fiona is kind.",
    "Fiona is round.",
    "Fiona is young.",
    "Gary is nice.",
    "Gary is round.",
    "If something is nice and big then it is kind.",
    "If Anne is round and Anne is young then Anne is green.",
    "Big things are nice.",
    "Kind things are white.",
    "If Anne is kind and Anne is round then Anne is young.",
    "If something is green and white then it is round.",
    "All round things are big."
  ],
  "_gold": "No",
  "_go


## Send to `/predict`

In [14]:
def to_payload(sample: dict) -> dict:
    body: dict = {"query_id": sample["query_id"], "type": sample["type"], "query": sample["query"]}
    if sample["premises"]:
        body["premises"] = sample["premises"]
    if sample["options"]:
        body["options"] = sample["options"]
    return body

async def call(client, sem, sample):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(PREDICT_URL, json=to_payload(sample), timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
    return {**sample, "_response": body, "_latency": time.perf_counter() - t0, "_error": err}

async def run_eval(samples):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(s):
            nonlocal done
            res = await call(client, sem, s)
            done += 1
            if done % 5 == 0 or done == len(samples):
                print(f"  {done}/{len(samples)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(s) for s in samples))

print(f"Sending {len(all_samples)} requests (concurrency={CONCURRENCY})...")
t0 = time.perf_counter()
results = await run_eval(all_samples)
wall = time.perf_counter() - t0
success = [r for r in results if not r["_error"]]
print(f"\nSuccess: {len(success)}/{len(results)}  |  errors: {len(results) - len(success)}  |  wall: {wall:.1f}s")

Sending 100 requests (concurrency=6)...
  100/100
Success: 99/100  |  errors: 1  |  wall: 374.1s


## Score (P1 answer + P2 proof-F1)

In [15]:
# ProofWriter (this subset) is CLOSED-WORLD: gold is only True/False (no
# Unknown), so "not provable" = No. Score our OWA outputs accordingly:
# True->Yes, False->No, Uncertain->No.  Set CLOSED_WORLD=False for OWA splits.
CLOSED_WORLD = True
_UNC = {"uncertain", "unknown"}

def _norm(a) -> str:
    s = str(a).strip().lower()
    if s == "true":
        return "yes"
    if s == "false":
        return "no"
    if s in _UNC:
        return "no" if CLOSED_WORLD else "uncertain"
    return s

def _f1(gold: list[int], pred: list[int]) -> float:
    g, p = set(gold), set(pred)
    if not g and not p:
        return 100.0
    if not g or not p:
        return 0.0
    inter = len(g & p)
    return 100.0 * 2 * inter / (len(g) + len(p)) if inter else 0.0

scored = []
for r in results:
    resp = r["_response"][0] if r["_response"] else None
    pred_ans = resp.get("answer") if resp else None
    pred_prem = resp.get("premises_used") if resp else None
    answer_ok = resp is not None and _norm(pred_ans) == _norm(r["_gold"])
    p1 = 100.0 if answer_ok else 0.0
    p2 = _f1(r["_gold_premises"], pred_prem or [])
    scored.append({
        "query_id": r["query_id"],
        "statement": r["_statement"],
        "gold_answer": r["_gold"],
        "pred_answer": pred_ans,
        "answer_ok": answer_ok,
        "gold_premises_used": r["_gold_premises"],
        "pred_premises_used": pred_prem,
        "p1_score": p1,
        "p2_score": round(p2, 2),
        "sample_score": round(0.5 * p1 + 0.5 * p2, 2),
        "explanation": resp.get("explanation") if resp else None,
        "fol": resp.get("fol") if resp else None,
        "latency_s": round(r["_latency"], 1),
        "error": r["_error"],
    })

print(f"{'sample':10s} {'gold':10s} {'pred':10s} {'P1':>4s} {'P2':>6s} {'score':>6s}")
print("-" * 52)
for x in scored:
    flag = "\u2713" if x["answer_ok"] else "\u2717"
    print(f"{x['query_id']:10s} {str(x['gold_answer']):10s} {str(x['pred_answer']):10s} "
          f"{x['p1_score']:>4.0f} {x['p2_score']:>6.1f} {x['sample_score']:>6.1f} {flag}")
    if x["error"]:
        print(f"   !! {x['error']}")

sample     gold       pred         P1     P2  score
----------------------------------------------------
PW_0000    No         False         0  100.0   50.0 ✗
PW_0001    Yes        Uncertain     0    0.0    0.0 ✗
PW_0002    No         No          100   66.7   83.3 ✓
PW_0003    No         False         0   33.3   16.7 ✗
PW_0004    No         False         0    0.0    0.0 ✗
PW_0005    No         No          100  100.0  100.0 ✓
PW_0006    Yes        Yes         100  100.0  100.0 ✓
PW_0007    No         No          100   66.7   83.3 ✓
PW_0008    Yes        No            0   28.6   14.3 ✗
PW_0009    Yes        No            0   40.0   20.0 ✗
PW_0010    No         No          100   57.1   78.6 ✓
PW_0011    Yes        Uncertain     0    0.0    0.0 ✗
PW_0012    No         Uncertain     0    0.0    0.0 ✗
PW_0013    Yes        Yes         100  100.0  100.0 ✓
PW_0014    No         Yes           0   57.1   28.6 ✗
PW_0015    Yes        Yes         100   80.0   90.0 ✓
PW_0016    No         Uncertain

## Summary + export results JSON

In [16]:
def _avg(xs):
    return round(sum(xs) / len(xs), 2) if xs else 0.0

def _block(rows):
    return {
        "n": len(rows),
        "answer_acc": _avg([100.0 if x["answer_ok"] else 0.0 for x in rows]),
        "p1_avg": _avg([x["p1_score"] for x in rows]),
        "p2_avg": _avg([x["p2_score"] for x in rows]),
        "sample_score_avg": _avg([x["sample_score"] for x in rows]),
    }

by_gold = {g: [x for x in scored if x["gold_answer"] == g] for g in ("Yes", "No", "Uncertain")}
summary = {
    "overall": _block(scored),
    "by_gold": {g: _block(rows) for g, rows in by_gold.items()},
    "errors": sum(1 for x in scored if x["error"]),
    "latency_mean_s": _avg([x["latency_s"] for x in scored]),
}

payload = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "endpoint": PREDICT_URL,
    "dataset": f"{HF_DATASET}:{HF_SPLIT}",
    "summary": summary,
    "results": scored,
}
OUTPUT_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False))

print("=== ProofWriter summary ===")
o = summary["overall"]
print(f"  overall  n={o['n']:<3d} answer_acc={o['answer_acc']:5.1f}%  "
      f"P1={o['p1_avg']:5.1f}  P2={o['p2_avg']:5.1f}  sample={o['sample_score_avg']:5.1f}")
for g, b in summary["by_gold"].items():
    print(f"  {g:9s} n={b['n']:<3d} answer_acc={b['answer_acc']:5.1f}%  sample={b['sample_score_avg']:5.1f}")
print(f"  errors={summary['errors']}  latency_mean={summary['latency_mean_s']}s")
print(f"\nwrote \u2192 {OUTPUT_JSON}")

=== ProofWriter summary ===
  overall  n=100 answer_acc= 32.0%  P1= 32.0  P2= 40.3  sample= 36.2
  Yes       n=53  answer_acc= 20.8%  sample= 30.1
  No        n=47  answer_acc= 44.7%  sample= 43.0
  Uncertain n=0   answer_acc=  0.0%  sample=  0.0
  errors=1  latency_mean=21.78s

wrote → /home/phuckhang/MyWorkspace/Exact2026/outputs/B08_proofwriter_eval_20260621_070848.json


## Notes
- **Local parquet cache:** the split's parquet shard(s) download once to `src/exact/datasets/proofwriter/<split>_NNNN.parquet` (HF CDN, no rate limit — the `/rows` API 429s on bulk paging). Read via pandas+pyarrow; later runs are offline/instant. Delete the file(s) to refresh. Requires `pyarrow` (`pip install pyarrow`).
- Sampling: `N_SAMPLES` rows drawn with `random.sample(seed=SEED)`; `N_SAMPLES=None` (or ≥ split size) → full split.
- Answers map `True→Yes`, `False→No`, `Unknown→Uncertain`. The `$question$` statement is wrapped as a polar query → YNU path.
- P2 compares gold `$proof$` sentence indices vs our `premises_used`. Proof granularity differs (ProofWriter lists every fact+rule; our Z3 core may be minimal) → treat P2 as indicative.
- `Unknown` items have an empty gold proof → a correctly-Uncertain answer with empty `premises_used` scores P2=100.
- Inspect one: `next(r for r in results if r['query_id'] == 'PW_0000')['_response']`.